In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, f1_score, average_precision_score
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
import warnings
import itertools
from sklearn.exceptions import ConvergenceWarning
import os
from sklearn.linear_model import Lasso, ElasticNet, Lars, LassoLars
from tqdm import tqdm
os.chdir('/home/bottia1/axel')

# Suppress convergence warnings in the main thread
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [7]:
# ---------------------------------------------------------
# Evaluation Metrics Helpers
# ---------------------------------------------------------
def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calc_rme(beta_true, beta_hat):
    # RME = ||B_hat - B_true|| / ||B_true||
    return np.linalg.norm(beta_hat - beta_true) / (np.linalg.norm(beta_true) + 1e-10)

def calc_rme_nonzeros(beta_true, beta_hat):
    mask = beta_true != 0
    return np.linalg.norm(beta_hat[mask] - beta_true[mask]) / (np.linalg.norm(beta_true[mask]) + 1e-10)

In [14]:
%%time
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Main Evaluation Loop (Diagnostics & Box Plots)
# ---------------------------------------------------------
datasets = ["Dataset_I", "Dataset_II", "Dataset_III", "Dataset_IV"]

# Pass the full list of your generated seeds so Pandas can calculate the spread
versions = ["v0", "v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8", "v9"] 

# The grid
alphas = [10.0, 1.0, 1e-1, 1e-2, 1e-3] 
l1_ratios = [0.5]

boxplot_data = []
final_scores = []

for name in datasets:
    print(f"\nProcessing {name}...")
    # THE FIX: Wrap versions in tqdm to generate a live progress bar
    for v in tqdm(versions, desc=f"{name} Seeds"):
        # Load X, y, AND the newly generated True Betas
        X = pd.read_csv(f'Simulation_Data/X_{name}_{v}.csv').values
        y = pd.read_csv(f'Simulation_Data/y_{name}_{v}.csv').values.ravel()
        true_betas = pd.read_csv(f'Simulation_Data/beta_{name}_{v}.csv').values.ravel()

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=(1/6), random_state=42)
        
        # Scale X globally once to save 30,000+ operations
        X_scaler = StandardScaler()
        X_train_scaled = X_scaler.fit_transform(X_train)
        X_val_scaled = X_scaler.transform(X_val)
        
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).ravel()
        
        # Build Truth Vector for F1/AUPRC Metrics
        p = X.shape[1]
        true_selected = (true_betas != 0).astype(int)

        # ==========================================
        # FULL GRID EVALUATION (All Models)
        # ==========================================
        for a in alphas:
            # 1. Basic LASSO
            l_model = LassoLars(alpha=a, fit_intercept=True, max_iter=2000)
            l_model.fit(X_train_scaled, y_train_scaled)
            l_pred = l_model.predict(X_val_scaled)
            
            # THE FIX: Actually calculate the RMSE!
            l_rmse = calc_rmse(y_val_scaled, l_pred)
            
            l_coef = l_model.coef_
            l_f1 = f1_score(true_selected, (np.abs(l_coef) > 1e-6).astype(int))
            l_auprc = average_precision_score(true_selected, np.abs(l_coef))

            # 2. Elastic Net
            e_model = ElasticNet(alpha=a, l1_ratio=0.5, fit_intercept=True, max_iter=2000, selection='random', tol=1e-3)
            e_model.fit(X_train_scaled, y_train_scaled)
            e_pred = e_model.predict(X_val_scaled)
            
            # THE FIX: Actually calculate the RMSE!
            e_rmse = calc_rmse(y_val_scaled, e_pred)
            
            e_coef = e_model.coef_
            e_f1 = f1_score(true_selected, (np.abs(e_coef) > 1e-4).astype(int))
            e_auprc = average_precision_score(true_selected, np.abs(e_coef))

            # 5. Store Row FOR THIS SPECIFIC ALPHA
            final_scores.append({
                "Dataset": name, "Seed": v, "Alpha": a,
                "L_RMSE": l_rmse, "L_F1": l_f1, "L_AUPRC": l_auprc, 
                "L_RME_All": calc_rme(true_betas, l_coef), "L_RME_Nz": calc_rme_nonzeros(true_betas, l_coef),
                
                "E_RMSE": e_rmse, "E_F1": e_f1, "E_AUPRC": e_auprc, 
                "E_RME_All": calc_rme(true_betas, e_coef), "E_RME_Nz": calc_rme_nonzeros(true_betas, e_coef)
            })

# Export Data
pd.DataFrame(final_scores).to_csv("master_base_metrics.csv", index=False)
print("\nComplete! All metrics saved to master_base_metrics.csv.")


Processing Dataset_I...


Dataset_I Seeds: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 15.36it/s]



Processing Dataset_II...


Dataset_II Seeds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  4.67it/s]



Processing Dataset_III...


Dataset_III Seeds: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:24<00:00,  2.43s/it]



Processing Dataset_IV...


Dataset_IV Seeds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:41<00:00,  4.11s/it]


Complete! All metrics saved to master_base_metrics.csv.
CPU times: user 41min 38s, sys: 2.31 s, total: 41min 40s
Wall time: 1min 8s


In [17]:
# ---------------------------------------------------------
# Final Output Comparison Table
# ---------------------------------------------------------
print("\n" + "="*60)
print("FINAL F1-SCORE COMPARISON")
print("="*60)

# Pandas makes a beautifully formatted console table
results_df = pd.DataFrame(final_scores)
print(results_df.to_string(index=False))

# ---------------------------------------------------------
# Final Averages & Standard Deviations (Paper Format)
# ---------------------------------------------------------
print("\n" + "="*60)
print("OPTIMAL F1-SCORES (± STD DEV)")
print("="*60)

# 1. Get the metrics corresponding to the LOWEST validation RMSE (Honest Selection)
summary_data = []
for base_name, group in results_df.groupby('Dataset'):
    row_dict = {"Dataset": base_name}
    
    # Locate the rows where L_RMSE is lowest for each seed
    l_best_rows = group.loc[group.groupby('Seed')['L_RMSE'].idxmin()]
    # Locate the rows where E_RMSE is lowest for each seed
    e_best_rows = group.loc[group.groupby('Seed')['E_RMSE'].idxmin()]
    
    # Format Basic LASSO (F1 and AUPRC)
    l_mean = l_best_rows['L_F1'].mean()
    l_std = l_best_rows['L_F1'].std()
    l_std_str = f"{l_std:.4f}" if pd.notna(l_std) else "0.0000"
    row_dict["L_F1"] = f"{l_mean:.4f} ± {l_std_str}"
    
    l_auprc_mean = l_best_rows['L_AUPRC'].mean()
    l_auprc_med = l_best_rows['L_AUPRC'].median()
    l_auprc_std = l_best_rows['L_AUPRC'].std()
    l_auprc_std_str = f"{l_auprc_std:.4f}" if pd.notna(l_auprc_std) else "0.0000"
    row_dict["L_AUPRC (Mean±Std | Med)"] = f"{l_auprc_mean:.4f} ± {l_auprc_std_str} | {l_auprc_med:.4f}"

    # Format Elastic Net (F1 and AUPRC)
    e_mean = e_best_rows['E_F1'].mean()
    e_std = e_best_rows['E_F1'].std()
    e_std_str = f"{e_std:.4f}" if pd.notna(e_std) else "0.0000"
    row_dict["E_F1"] = f"{e_mean:.4f} ± {e_std_str}"
    
    e_auprc_mean = e_best_rows['E_AUPRC'].mean()
    e_auprc_med = e_best_rows['E_AUPRC'].median()
    e_auprc_std = e_best_rows['E_AUPRC'].std()
    e_auprc_std_str = f"{e_auprc_std:.4f}" if pd.notna(e_auprc_std) else "0.0000"
    row_dict["E_AUPRC (Mean±Std | Med)"] = f"{e_auprc_mean:.4f} ± {e_auprc_std_str} | {e_auprc_med:.4f}"
        
    summary_data.append(row_dict)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))


FINAL F1-SCORE COMPARISON
    Dataset Seed  Alpha   L_RMSE     L_F1  L_AUPRC  L_RME_All  L_RME_Nz   E_RMSE     E_F1  E_AUPRC  E_RME_All  E_RME_Nz
  Dataset_I   v0 10.000 1.423785 0.000000 0.100000   1.000000  1.000000 1.423785 0.000000 0.100000   1.000000  1.000000
  Dataset_I   v0  1.000 1.423785 0.000000 0.100000   1.000000  1.000000 0.964445 0.571429 0.460000   0.980883  0.980883
  Dataset_I   v0  0.100 0.387925 0.533333 0.440000   0.950331  0.950320 0.372165 0.400000 0.533333   0.948082  0.947989
  Dataset_I   v0  0.010 0.281222 0.410256 0.820000   0.885166  0.884179 0.284083 0.320000 0.820000   0.881276  0.879589
  Dataset_I   v0  0.001 0.360915 0.307692 0.787500   0.860960  0.857262 0.443619 0.238806 0.662857   0.895024  0.890148
  Dataset_I   v1 10.000 0.769542 0.000000 0.100000   1.000000  1.000000 0.769542 0.000000 0.100000   1.000000  1.000000
  Dataset_I   v1  1.000 0.769542 0.000000 0.100000   1.000000  1.000000 0.518345 0.571429 0.460000   0.978590  0.978590
  Dataset_I  